In [1]:
import os
import imageio
import matplotlib.pyplot as plt
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import RadarPointCloud
from nuscenes.utils.geometry_utils import transform_matrix
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud
import matplotlib.pyplot as plt
from PIL import Image
import io
from tqdm import tqdm
import numpy as np



# Initialize nuScenes
dataroot = 'G:\\nuscenes full\\v1.0-trainval01_blobs'

nusc = NuScenes(version='v1.0-trainval', dataroot=dataroot, verbose=True)
# Get scene 1
scene = nusc.scene[0]
first_sample_token = scene['first_sample_token']
sample = nusc.get('sample', first_sample_token)

# Create a folder to store radar plots
os.makedirs('radar_frames', exist_ok=True)
frame_paths = []

# Loop through all samples in the scene
frame_idx = 0
while sample:
    radar_data = nusc.get('sample_data', sample['data']['RADAR_FRONT'])
    pcl = RadarPointCloud.from_file(os.path.join(nusc.dataroot, radar_data['filename']))
    
    # Extract x, y for 2D plotting (vehicle coordinate frame)
    points = pcl.points[:2, :]  # x, y

    # Plot
    plt.figure(figsize=(6, 6))
    plt.scatter(points[0, :], points[1, :], s=5, c='red', alpha=0.7)
    plt.title(f'RADAR_FRONT - Frame {frame_idx}')
    plt.xlabel('X [m]')
    plt.ylabel('Y [m]')
    plt.grid(True)
    plt.axis('equal')

    # Save frame
    frame_path = f'radar_frames/frame_{frame_idx:03d}.png'
    plt.savefig(frame_path)
    frame_paths.append(frame_path)
    plt.close()

    # Move to next sample
    if sample['next']:
        sample = nusc.get('sample', sample['next'])
        frame_idx += 1
    else:
        break

# Generate GIF
with imageio.get_writer('radar_front_scene1.gif', mode='I', duration=0.3) as writer:
    for frame_path in frame_paths:
        image = imageio.imread(frame_path)
        writer.append_data(image)

print("GIF saved as radar_front_scene1.gif")


Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 44.620 seconds.
Reverse indexing ...
Done reverse indexing in 6.2 seconds.


C:\Users\Dariush\AppData\Local\Temp\ipykernel_23992\2263013789.py:64: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(frame_path)


GIF saved as radar_front_scene1.gif


In [2]:


# Get scene 1
scene = nusc.scene[0]  # scene-0001

# Get the first sample in the scene
sample_token = scene['first_sample_token']
sample = nusc.get('sample', sample_token)

# Get lidar top data
lidar_token = sample['data']['LIDAR_TOP']

# Container for images
frames = []

while lidar_token:
    # Get the sample_data record
    lidar_data = nusc.get('sample_data', lidar_token)
    
    # Load point cloud
    pcl_path = os.path.join(nusc.dataroot, lidar_data['filename'])
    pcl = LidarPointCloud.from_file(pcl_path)
    
    # Plot as BEV using matplotlib
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(pcl.points[0], pcl.points[1], s=0.1, c='black')
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    ax.set_title(f"Timestamp: {lidar_data['timestamp']}")
    ax.axis('off')

    # Save the plot to a PIL image
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    image = Image.open(buf)
    frames.append(image)

    # Move to next lidar frame
    lidar_token = lidar_data['next']

# Save as GIF
gif_path = 'scene1_lidar_bev.gif'
frames[0].save(gif_path, format='GIF', save_all=True, append_images=frames[1:], duration=0.1, loop=0)

print(f"GIF saved to {gif_path}")


GIF saved to scene1_lidar_bev.gif


In [4]:


def lidar_to_bev(lidar_points, x_range=(-50, 50), y_range=(-50, 50), grid_size=0.2):
    x_min, x_max = x_range
    y_min, y_max = y_range
    resolution = grid_size

    # Filter points in range
    mask = (lidar_points[0] >= x_min) & (lidar_points[0] <= x_max) & \
           (lidar_points[1] >= y_min) & (lidar_points[1] <= y_max)
    lidar_points = lidar_points[:, mask]

    # Convert to grid coordinates
    x_grid = np.floor((lidar_points[0] - x_min) / resolution).astype(int)
    y_grid = np.floor((lidar_points[1] - y_min) / resolution).astype(int)

    # Create occupancy grid
    bev_map = np.zeros((int((x_max - x_min) / resolution), int((y_max - y_min) / resolution)))
    bev_map[x_grid, y_grid] = 1  # Mark occupied cells

    return bev_map
# Main function to generate BEV

    
    
    
    
# Initialize nuScenes
dataroot = 'G:\\nuscenes full\\v1.0-trainval01_blobs'

nusc = NuScenes(version='v1.0-trainval', dataroot=dataroot, verbose=True)



# Choose scene 1
scene = nusc.scene[0]  # scene[0] is scene-0001 in v1.0-mini
first_sample_token = scene['first_sample_token']
sample = nusc.get('sample', first_sample_token)

# Output directory
output_dir = 'scene1_images'
os.makedirs(output_dir, exist_ok=True)

# Collect images
images = []
limages=[]
frame_idx = 0

while sample:
    
    cam_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
    image_path = os.path.join(nusc.dataroot, cam_data['filename'])
    
    lidar_data = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
    lidar_path = os.path.join(nusc.dataroot, lidar_data["filename"])

    # Load LiDAR point cloud
    pc = LidarPointCloud.from_file(lidar_path)
    bev_map = lidar_to_bev(pc.points)
    limages.append(bev_map)
    # Load and resize (optional)
    img = Image.open(image_path).resize((640, 360))  # resize to keep GIF size small
    output_img_path = os.path.join(output_dir, f"{frame_idx:04d}.jpg")
    img.save(output_img_path)
    images.append(imageio.imread(output_img_path))

    # Go to next frame
    if cam_data['next'] == '':
        break
    sample = nusc.get('sample', sample['next'])
    frame_idx += 1

# Save GIF
gif_path = 'scene1_cam_front.gif'
lgif_path = 'scene1_lidartop.gif'
imageio.mimsave(gif_path, images, duration=10)  # adjust duration for speed
#imageio.mimsave(lgif_path, limages, duration=0.3)  # adjust duration for speed

print(f"Saved GIF: {gif_path}")


Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 44.485 seconds.
Reverse indexing ...
Done reverse indexing in 6.0 seconds.


C:\Users\Dariush\AppData\Local\Temp\ipykernel_23992\3178618893.py:63: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(output_img_path))


Saved GIF: scene1_cam_front.gif


In [12]:
gif_path = 'sssscene1_cam_front.gif'

imageio.mimsave(gif_path, images, duration=1000)  # adjust duration for speed


In [13]:
gif_path = 'ssscene1_lidar_bev.gif'
frames[0].save(gif_path, format='GIF', save_all=True, append_images=frames[1:], duration=0.00001)